In [1]:
from __future__ import annotations

import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery, QueryReference

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
for collection_name in ["HardwareReview", "HardwareProduct"]:
    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)
        print("Deleted:", collection_name)

In [5]:
products = client.collections.create(
    name="HardwareProduct",
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    properties=[
        wvc.config.Property(
            name="name",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="component",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="description",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created HardwareProduct")

Created HardwareProduct


In [9]:
reviews = client.collections.create(
    name="HardwareReview",
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    properties=[
        wvc.config.Property(
            name="review_text",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="rating",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="reviewer",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
    references=[
        wvc.config.ReferenceProperty(
            name="forProduct",
            target_collection="HardwareProduct",
        ),
    ],
)

print("Created HardwareReview")

Created HardwareReview


In [11]:
import os
from getpass import getpass

os.environ.pop("OPENAI_API_KEY", None)
os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [13]:
client.close()

In [14]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [15]:
products = client.collections.get("HardwareProduct")
reviews = client.collections.get("HardwareReview")

In [16]:
product_data = [
    {
        "name": "NVIDIA RTX 4000 Ada",
        "component": "GPU",
        "description": "Professional GPU for AI workloads, CUDA, rendering and local machine learning experiments.",
    },
    {
        "name": "Samsung 990 PRO NVMe SSD",
        "component": "Storage",
        "description": "Fast NVMe storage for datasets, Docker images, virtual machines and large project files.",
    },
    {
        "name": "64 GB DDR5 RAM Kit",
        "component": "RAM",
        "description": "Large memory kit for data science notebooks, Docker containers and virtual machines.",
    },
]

product_uuids = {}

for product in product_data:
    uuid = products.data.insert(product)
    product_uuids[product["name"]] = uuid
    print(product["name"], "->", uuid)

NVIDIA RTX 4000 Ada -> cc020c24-1ddc-40b0-9428-aebc9025eb07
Samsung 990 PRO NVMe SSD -> 156d2d9c-f3f7-4c53-a63f-6b5596e9a8ae
64 GB DDR5 RAM Kit -> d105fe7c-39b1-4e2d-8473-bf363a9125cf


In [17]:
review_data = [
    {
        "review_text": "Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.",
        "rating": 5,
        "reviewer": "Piotr",
        "product_name": "NVIDIA RTX 4000 Ada",
    },
    {
        "review_text": "Excellent drive for development work. Docker images and datasets load very quickly.",
        "rating": 5,
        "reviewer": "Anna",
        "product_name": "Samsung 990 PRO NVMe SSD",
    },
    {
        "review_text": "Useful memory upgrade for running several virtual machines and Jupyter notebooks at the same time.",
        "rating": 4,
        "reviewer": "Mark",
        "product_name": "64 GB DDR5 RAM Kit",
    },
    {
        "review_text": "This graphics card is a strong choice for CUDA workloads and AI prototyping.",
        "rating": 5,
        "reviewer": "Developer",
        "product_name": "NVIDIA RTX 4000 Ada",
    },
]

In [18]:
review_uuids = []

for review_item in review_data:
    product_name = review_item["product_name"]

    review_properties = {
        "review_text": review_item["review_text"],
        "rating": review_item["rating"],
        "reviewer": review_item["reviewer"],
    }

    review_uuid = reviews.data.insert(review_properties)
    review_uuids.append(review_uuid)

    reviews.data.reference_add(
        from_uuid=review_uuid,
        from_property="forProduct",
        to=product_uuids[product_name],
    )

    print("Review:", review_uuid)
    print("Linked to product:", product_name)
    print("-" * 80)

Review: c07cada9-3547-4591-ac8a-73ed7691030e
Linked to product: NVIDIA RTX 4000 Ada
--------------------------------------------------------------------------------
Review: 2a1f5550-7b27-42e6-b5f0-561f4089b2b5
Linked to product: Samsung 990 PRO NVMe SSD
--------------------------------------------------------------------------------
Review: e7b80cb5-964a-4cc0-a9f7-cce56ed7217c
Linked to product: 64 GB DDR5 RAM Kit
--------------------------------------------------------------------------------
Review: b1a133c8-72f1-4539-8164-256c1829ad97
Linked to product: NVIDIA RTX 4000 Ada
--------------------------------------------------------------------------------


In [19]:
response = reviews.query.fetch_objects(
    limit=10,
    return_properties=["review_text", "rating", "reviewer"],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Review:", obj.properties["review_text"])
    print("Rating:", obj.properties["rating"])
    print("Reviewer:", obj.properties["reviewer"])
    print("-" * 80)

UUID: 2a1f5550-7b27-42e6-b5f0-561f4089b2b5
Review: Excellent drive for development work. Docker images and datasets load very quickly.
Rating: 5
Reviewer: Anna
--------------------------------------------------------------------------------
UUID: b1a133c8-72f1-4539-8164-256c1829ad97
Review: This graphics card is a strong choice for CUDA workloads and AI prototyping.
Rating: 5
Reviewer: Developer
--------------------------------------------------------------------------------
UUID: c07cada9-3547-4591-ac8a-73ed7691030e
Review: Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.
Rating: 5
Reviewer: Piotr
--------------------------------------------------------------------------------
UUID: e7b80cb5-964a-4cc0-a9f7-cce56ed7217c
Review: Useful memory upgrade for running several virtual machines and Jupyter notebooks at the same time.
Rating: 4
Reviewer: Mark
--------------------------------------------------------------------------------


In [20]:
from weaviate.classes.query import QueryReference

response = reviews.query.fetch_objects(
    limit=10,
    return_properties=["review_text", "rating", "reviewer"],
    return_references=[
        QueryReference(
            link_on="forProduct",
            return_properties=["name", "component", "description"],
        )
    ],
)

for obj in response.objects:
    print("Review:", obj.properties["review_text"])
    print("Rating:", obj.properties["rating"])
    print("Reviewer:", obj.properties["reviewer"])

    product_refs = obj.references["forProduct"].objects

    for product_ref in product_refs:
        print("Product:", product_ref.properties["name"])
        print("Component:", product_ref.properties["component"])
        print("Description:", product_ref.properties["description"])

    print("-" * 80)

Review: Excellent drive for development work. Docker images and datasets load very quickly.
Rating: 5
Reviewer: Anna
Product: Samsung 990 PRO NVMe SSD
Component: Storage
Description: Fast NVMe storage for datasets, Docker images, virtual machines and large project files.
--------------------------------------------------------------------------------
Review: This graphics card is a strong choice for CUDA workloads and AI prototyping.
Rating: 5
Reviewer: Developer
Product: NVIDIA RTX 4000 Ada
Component: GPU
Description: Professional GPU for AI workloads, CUDA, rendering and local machine learning experiments.
--------------------------------------------------------------------------------
Review: Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.
Rating: 5
Reviewer: Piotr
Product: NVIDIA RTX 4000 Ada
Component: GPU
Description: Professional GPU for AI workloads, CUDA, rendering and local machine learning experiments.
---------------------------

In [21]:
response = reviews.query.near_text(
    query="good hardware for AI and CUDA workloads",
    limit=3,
    return_properties=["review_text", "rating", "reviewer"],
    return_references=[
        QueryReference(
            link_on="forProduct",
            return_properties=["name", "component"],
        )
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Review:", obj.properties["review_text"])
    print("Rating:", obj.properties["rating"])
    print("Reviewer:", obj.properties["reviewer"])

    for product_ref in obj.references["forProduct"].objects:
        print("Product:", product_ref.properties["name"])
        print("Component:", product_ref.properties["component"])

    print("-" * 80)

Distance: 0.2766609191894531
Review: This graphics card is a strong choice for CUDA workloads and AI prototyping.
Rating: 5
Reviewer: Developer
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------
Distance: 0.41541457176208496
Review: Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.
Rating: 5
Reviewer: Piotr
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------
Distance: 0.5386828184127808
Review: Excellent drive for development work. Docker images and datasets load very quickly.
Rating: 5
Reviewer: Anna
Product: Samsung 990 PRO NVMe SSD
Component: Storage
--------------------------------------------------------------------------------


In [22]:
response = reviews.query.near_text(
    query="fast disk for datasets and Docker images",
    limit=3,
    return_properties=["review_text", "rating", "reviewer"],
    return_references=[
        QueryReference(
            link_on="forProduct",
            return_properties=["name", "component"],
        )
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Review:", obj.properties["review_text"])
    print("Rating:", obj.properties["rating"])
    print("Reviewer:", obj.properties["reviewer"])

    for product_ref in obj.references["forProduct"].objects:
        print("Product:", product_ref.properties["name"])
        print("Component:", product_ref.properties["component"])

    print("-" * 80)

Distance: 0.37635791301727295
Review: Excellent drive for development work. Docker images and datasets load very quickly.
Rating: 5
Reviewer: Anna
Product: Samsung 990 PRO NVMe SSD
Component: Storage
--------------------------------------------------------------------------------
Distance: 0.6392411589622498
Review: Useful memory upgrade for running several virtual machines and Jupyter notebooks at the same time.
Rating: 4
Reviewer: Mark
Product: 64 GB DDR5 RAM Kit
Component: RAM
--------------------------------------------------------------------------------
Distance: 0.6980149745941162
Review: Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.
Rating: 5
Reviewer: Piotr
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------


In [23]:
from weaviate.classes.query import Filter

response = reviews.query.near_text(
    query="hardware for machine learning",
    filters=Filter.by_property("rating").greater_or_equal(5),
    limit=3,
    return_properties=["review_text", "rating", "reviewer"],
    return_references=[
        QueryReference(
            link_on="forProduct",
            return_properties=["name", "component"],
        )
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Rating:", obj.properties["rating"])
    print("Review:", obj.properties["review_text"])
    print("Reviewer:", obj.properties["reviewer"])

    for product_ref in obj.references["forProduct"].objects:
        print("Product:", product_ref.properties["name"])
        print("Component:", product_ref.properties["component"])

    print("-" * 80)

Distance: 0.5113213658332825
Rating: 5
Review: Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.
Reviewer: Piotr
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------
Distance: 0.516858696937561
Rating: 5
Review: This graphics card is a strong choice for CUDA workloads and AI prototyping.
Reviewer: Developer
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------
Distance: 0.5956108570098877
Rating: 5
Review: Excellent drive for development work. Docker images and datasets load very quickly.
Reviewer: Anna
Product: Samsung 990 PRO NVMe SSD
Component: Storage
--------------------------------------------------------------------------------


In [24]:
def search_reviews(query: str, *, limit: int = 3) -> None:
    response = reviews.query.near_text(
        query=query,
        limit=limit,
        return_properties=["review_text", "rating", "reviewer"],
        return_references=[
            QueryReference(
                link_on="forProduct",
                return_properties=["name", "component"],
            )
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    print("QUERY:", query)
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Review:", obj.properties["review_text"])
        print("Rating:", obj.properties["rating"])
        print("Reviewer:", obj.properties["reviewer"])

        for product_ref in obj.references["forProduct"].objects:
            print("Product:", product_ref.properties["name"])
            print("Component:", product_ref.properties["component"])

        print("-" * 80)

In [25]:
search_reviews("AI workstation GPU")

QUERY: AI workstation GPU
--------------------------------------------------------------------------------
Distance: 0.4412901997566223
Review: Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.
Rating: 5
Reviewer: Piotr
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------
Distance: 0.4621995687484741
Review: This graphics card is a strong choice for CUDA workloads and AI prototyping.
Rating: 5
Reviewer: Developer
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------
Distance: 0.592225968837738
Review: Useful memory upgrade for running several virtual machines and Jupyter notebooks at the same time.
Rating: 4
Reviewer: Mark
Product: 64 GB DDR5 RAM Kit
Component: RAM
--------------------------------------------------------------------------------


In [26]:
search_reviews("fast storage for development")

QUERY: fast storage for development
--------------------------------------------------------------------------------
Distance: 0.4751685857772827
Review: Excellent drive for development work. Docker images and datasets load very quickly.
Rating: 5
Reviewer: Anna
Product: Samsung 990 PRO NVMe SSD
Component: Storage
--------------------------------------------------------------------------------
Distance: 0.607206404209137
Review: Useful memory upgrade for running several virtual machines and Jupyter notebooks at the same time.
Rating: 4
Reviewer: Mark
Product: 64 GB DDR5 RAM Kit
Component: RAM
--------------------------------------------------------------------------------
Distance: 0.6806544661521912
Review: Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.
Rating: 5
Reviewer: Piotr
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------


In [27]:
search_reviews("memory for virtual machines")

QUERY: memory for virtual machines
--------------------------------------------------------------------------------
Distance: 0.3846582770347595
Review: Useful memory upgrade for running several virtual machines and Jupyter notebooks at the same time.
Rating: 4
Reviewer: Mark
Product: 64 GB DDR5 RAM Kit
Component: RAM
--------------------------------------------------------------------------------
Distance: 0.6737166047096252
Review: Great GPU for local AI experiments. The VRAM is very helpful when working with larger models.
Rating: 5
Reviewer: Piotr
Product: NVIDIA RTX 4000 Ada
Component: GPU
--------------------------------------------------------------------------------
Distance: 0.7497435808181763
Review: Excellent drive for development work. Docker images and datasets load very quickly.
Rating: 5
Reviewer: Anna
Product: Samsung 990 PRO NVMe SSD
Component: Storage
--------------------------------------------------------------------------------


In [28]:
response = reviews.query.fetch_objects(
    limit=1,
    return_properties=["review_text", "rating", "reviewer"],
    return_references=[
        QueryReference(
            link_on="forProduct",
            return_properties=["name", "component", "description"],
        )
    ],
)

obj = response.objects[0]

print("Review UUID:", obj.uuid)
print("Review properties:")
print(obj.properties)

print("\nReferences:")
print(obj.references)

Review UUID: 2a1f5550-7b27-42e6-b5f0-561f4089b2b5
Review properties:
{'reviewer': 'Anna', 'rating': 5, 'review_text': 'Excellent drive for development work. Docker images and datasets load very quickly.'}

References:
{'forProduct': <weaviate.collections.classes.internal._CrossReference object at 0x73081d45c0d0>}


In [29]:
client.close()